# RQ2-v3 development: Geo-HT vs Resource-HT — seed 3, T4×2

Two from-scratch 100-epoch runs optimize the same dense-family target with identical model/data/loss/compute protocol. Each batch contains endpoints `.25,1.0` and two distinct sampled interior widths. The only algorithmic difference is the frozen proposal `(pi,q)`; exact Horvitz–Thompson weights `(1/7)/pi_i` make both estimators unbiased for the same objective. This is development evidence, not confirmatory inference.

In [ ]:
import os, subprocess, sys, json, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1'], check=True)
import torch
assert torch.cuda.device_count() == 2, f'Select Kaggle T4 x2; detected {torch.cuda.device_count()} GPU(s)'
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])

## Resolve development evidence and the passed variance gate

In [ ]:
import importlib
import rq2_anchor_placement, rq2_dynamic_seed3_pilot, rq2_ht_seed3
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_dynamic_seed3_pilot = importlib.reload(rq2_dynamic_seed3_pilot)
rq2_ht_seed3 = importlib.reload(rq2_ht_seed3)
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(
    RQ2_INPUT, '/kaggle/working/materialized-rq2-ht-seed3'
)
GATE_PATH = rq2_ht_seed3.find_variance_gate(
    Path('/kaggle/input'), '/kaggle/working/materialized-v3-variance-gate'
)
GATE_BUNDLE_ROOT = GATE_PATH.parent.parent
THEORY_ROOT, PREVIEW_ROOT = rq2_dynamic_seed3_pilot.find_frozen_policy_roots(
    GATE_BUNDLE_ROOT, '/kaggle/working/materialized-v3-frozen-policies'
)
gate = json.loads(GATE_PATH.read_text())
print('RQ2 root:', RQ2_ROOT)
print('Variance gate:', GATE_PATH, gate['decision'])
print('Frozen theory policy:', THEORY_ROOT)
print('Frozen resource policy:', PREVIEW_ROOT)

## Run both HT methods concurrently
GPU 0 and GPU 1 each run one method. Training is `50+50` with optimizer/scheduler reset at epoch 51. Model snapshots and dense validation are produced at epochs 10,20,…,100. All evaluation uses only the fixed 5k validation split after BN recalibration.

In [ ]:
import scripts.run_ht_seed3 as ht_runner
ht_runner = importlib.reload(ht_runner)
RUN_DIR = Path('/kaggle/working/rq2-v3-ht-seed3')
started = time.perf_counter()
result = ht_runner.run_ht_development(
    development_root=RQ2_ROOT,
    theory_root=THEORY_ROOT,
    preview_root=PREVIEW_ROOT,
    variance_gate_path=GATE_PATH,
    output_dir=RUN_DIR,
    gpu_ids=[0,1],
)
print(f"HT development completed in {(time.perf_counter()-started)/3600:.2f} hours")
print(json.dumps(result['decision'], indent=2))

## Inspect HT sanity, final accuracy, and convergence

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(RUN_DIR/'pretraining_policy_checks.csv'))
sanity = pd.read_csv(RUN_DIR/'pretraining_ht_sanity.csv')
display(sanity.groupby('method').agg(
    max_pi_error=('pi_error', lambda x: x.abs().max()),
    max_effective_weight_error=('effective_weight_error', lambda x: x.abs().max()),
))
display(pd.read_csv(RUN_DIR/'checkpoint_family_summary.csv'))
display(pd.read_csv(RUN_DIR/'ht_convergence_comparison.csv'))
display(pd.read_csv(RUN_DIR/'ht_convergence_common_compute.csv'))
display(pd.read_csv(RUN_DIR/'final_width_comparison.csv'))
display(Image(filename=str(RUN_DIR/'ht_convergence_vs_epoch.png')))
display(Image(filename=str(RUN_DIR/'ht_convergence_vs_compute.png')))
display(Image(filename=str(RUN_DIR/'final_width_delta.png')))

## Validate and export the development bundle

In [ ]:
required = [
    RUN_DIR/'ht_development_decision.json',
    RUN_DIR/'pretraining_policy_checks.csv', RUN_DIR/'pretraining_ht_sanity.csv',
    RUN_DIR/'dense_validation_by_checkpoint.csv', RUN_DIR/'checkpoint_family_summary.csv',
    RUN_DIR/'ht_convergence_comparison.csv', RUN_DIR/'ht_convergence_common_compute.csv',
    RUN_DIR/'final_width_comparison.csv',
]
for method in ['geo_ht','resource_ht']:
    method_root = RUN_DIR/method/'seed_3'
    required += [method_root/f'epoch_{epoch:03d}.pt' for epoch in range(10,101,10)]
    required += [method_root/'training_epoch_metrics.csv', method_root/'ht_width_metrics_by_epoch.csv',
                 method_root/'pair_counts_by_epoch.csv', method_root/'training_provenance.json']
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
assert not missing, f'Missing HT artifacts: {missing}'
decision = json.loads((RUN_DIR/'ht_development_decision.json').read_text())
assert decision['test_used'] is False and decision['same_target_objective'] is True
bundle_path = Path('/kaggle/working/rq2-v3-ht-seed3.zip')
excluded = {'latest.pt','resumable_final.pt'}
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
    for path in RUN_DIR.rglob('*'):
        if path.is_file() and path.name not in excluded:
            bundle.write(path, path.relative_to(RUN_DIR))
print('Development decision:', decision)
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**30:.2f} GiB')
bundle_path